# D221 - HDFS in Depth

A hands-on guide to the Hadoop Distributed File System on the single-node Hadoop 3.3.6 training cluster. This lab focuses on the `hdfs dfs` command family, file blocks, permissions, common directory conventions, and the roles of the NameNode and DataNode.
> For beginners, run command from Linux prompt, not from notebook
> Run this notebook from Jupyter inside Ubuntu/WSL. Start HDFS before beginning. Code cells use `%%bash`.

## 1. HDFS mental model

HDFS presents one filesystem namespace, but stores file contents as large blocks.

- The **NameNode** owns namespace metadata: paths, directory entries, owners, permissions, replication targets, and the mapping from files to blocks.
- **DataNodes** store the actual block bytes, serve reads and writes, create/delete replicas, and send heartbeats and block reports.
- An **HDFS client** asks the NameNode where blocks belong, then transfers data directly to or from DataNodes. File contents do not normally flow through the NameNode.
- The **SecondaryNameNode** periodically checkpoints NameNode metadata. It is not a standby NameNode and does not store user file blocks.

### What happens during common commands?

| Operation | NameNode work | DataNode work |
|---|---|---|
| `-ls`, `-stat`, `-mkdir` | Resolve/read/change namespace metadata | Usually none |
| `-put`, `-appendToFile` | Authorize path and choose block targets | Receive and store bytes |
| `-cat`, `-get` | Return block locations and authorize access | Send bytes to client |
| `-mv` inside HDFS | Rename metadata entry | Usually no block transfer |
| `-cp` inside HDFS | Create destination metadata and blocks | Read and write block bytes |
| `-rm` | Remove namespace entry and schedule blocks for deletion | Delete blocks when instructed |
| `-chmod`, `-chown` | Change metadata | None |

A successful `-mkdir` proves the NameNode works. A successful `-put` or `-cat` also proves that a DataNode can transfer data.

## 2. Pre-flight checks

Confirm the HDFS URI, running daemons, safe mode, and live DataNode count.

In [ ]:
%%bash
echo "User: $USER"
echo "Default filesystem: $(hdfs getconf -confKey fs.defaultFS)"
jps
hdfs dfsadmin -safemode get
hdfs dfsadmin -report | grep -E 'Live datanodes|Dead datanodes|Configured Capacity|DFS Used%'

## 3. Learn the command-line interface

`hdfs dfs` is the preferred HDFS shell entry point. `hadoop fs` is a generic filesystem shell that can address HDFS and other Hadoop-compatible filesystems. With `fs.defaultFS=hdfs://localhost:9000`, both normally reach this cluster.

In [ ]:
%%bash
hdfs dfs -usage ls
echo
hdfs dfs -help mkdir | head -20
echo
hdfs dfs -help | head -35

## 4. Paths and Hadoop directory conventions

These are conventions, not mandatory partitions:

| HDFS path | Typical purpose |
|---|---|
| `/` | HDFS namespace root |
| `/user` | Parent for user home directories |
| `/user/$USER` | Current user's HDFS home; relative HDFS paths start here |
| `/tmp` | Shared temporary/staging area; permissions and cleanup policy matter |
| `/user/hive/warehouse` | Common default Hive managed-table warehouse |
| `/tmp/hive` | Common Hive scratch/staging area |
| `/apps`, `/data`, `/warehouse` | Organization-specific application or curated data areas |

HDFS `/tmp` is not Linux `/tmp`. `/user/$USER` in HDFS is not `$HOME` on the local filesystem.

In [ ]:
%%bash
echo 'HDFS root:'
hdfs dfs -ls /
echo
echo 'HDFS /user:'
hdfs dfs -ls /user 2>/dev/null || true
echo
echo 'Local Linux home:'
ls -ld "$HOME"

A relative HDFS path such as `d221/input` resolves under `/user/$USER`. Use absolute paths in shared scripts because they make the target obvious.

In [ ]:
%%bash
echo "HDFS home from configuration: $(hdfs dfs -getHomeDirectory)"
echo "HDFS working path: /user/$USER"
hdfs dfs -ls . 2>/dev/null || echo 'Create the HDFS home directory in the next section.'

## 5. Create the lab directory tree

`-mkdir -p` creates missing parents and succeeds if the directory already exists. Directory creation changes NameNode metadata; it does not write a user-data block to a DataNode.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -mkdir -p "/user/$USER"
hdfs dfs -mkdir -p "$LAB/input/raw" "$LAB/input/reference" "$LAB/output" "$LAB/archive"
hdfs dfs -ls -R "$LAB"

## 6. Prepare local files and a local directory

These commands create ordinary Linux files. They do not touch HDFS yet.

In [ ]:
%%bash
LOCAL_DIR="/tmp/d221-local"
mkdir -p "$LOCAL_DIR/batch"
printf '%s\n' '101,Asha,Engineering' '102,Bala,Finance' '103,Chen,Engineering' > "$LOCAL_DIR/employees.csv"
printf '%s\n' 'hdfs stores blocks' 'namenode stores metadata' > "$LOCAL_DIR/batch/part-01.txt"
printf '%s\n' 'datanodes store bytes' 'clients stream data' > "$LOCAL_DIR/batch/part-02.txt"
find "$LOCAL_DIR" -maxdepth 2 -type f -printf '%p  %s bytes\n'

## 7. Upload files: `-put` and `-copyFromLocal`

Both commands copy local bytes into HDFS. The client requests block targets from the NameNode and streams bytes to a DataNode pipeline. `-f` overwrites an existing destination file.

In [ ]:
%%bash
LAB="/user/$USER/d221"
LOCAL_DIR="/tmp/d221-local"
hdfs dfs -put -f "$LOCAL_DIR/employees.csv" "$LAB/input/raw/"
hdfs dfs -copyFromLocal -f "$LOCAL_DIR/batch/part-01.txt" "$LAB/input/raw/"
hdfs dfs -ls -h "$LAB/input/raw"

Upload a complete local directory. The directory itself becomes a child of the destination.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -put -f /tmp/d221-local/batch "$LAB/input/"
hdfs dfs -ls -R "$LAB/input"

### Other upload forms

- `hdfs dfs -put file1 file2 DEST_DIR` uploads multiple sources.
- `hdfs dfs -put -` reads standard input and creates a destination file.
- `hdfs dfs -moveFromLocal SOURCE DEST` uploads and then removes the local source after success. Use it only when that deletion is intended.
- `hdfs dfs -put -l SOURCE DEST` requests lazy-persist storage when the cluster supports it.
- `hdfs dfs -put -d SOURCE DEST` skips creation of the temporary `._COPYING_` file. The normal temporary-file behavior is safer for readers.

In [ ]:
%%bash
LAB="/user/$USER/d221"
printf '%s\n' 'created through standard input' | hdfs dfs -put -f - "$LAB/input/raw/stdin.txt"
hdfs dfs -ls "$LAB/input/raw/stdin.txt"

## 8. List and inspect namespace entries

`-ls` displays type/permissions, replication, owner, group, size, modification time, and path. A directory shows replication as `-` because replication applies to file blocks.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -ls "$LAB/input"
echo
hdfs dfs -ls -h -R "$LAB/input"
echo
hdfs dfs -find "$LAB" -name '*.txt' -print

`-stat` is useful in scripts because its output format is controllable. `%n` name, `%b` bytes, `%o` block size, `%r` replication, `%u` owner, `%g` group, `%a` octal permissions, `%y` modification time.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/employees.csv"
hdfs dfs -stat 'name=%n bytes=%b block_size=%o replication=%r owner=%u group=%g mode=%a modified=%y' "$FILE"

`-test` prints nothing; its exit status answers a question. It is intended for shell conditions.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/employees.csv"
hdfs dfs -test -e "$FILE" && echo 'exists'
hdfs dfs -test -f "$FILE" && echo 'is a file'
hdfs dfs -test -d "/user/$USER/d221/input" && echo 'is a directory'
hdfs dfs -test -s "$FILE" && echo 'has non-zero length'

## 9. Read data without downloading a file

`-cat`, `-head`, and `-tail` stream bytes from DataNodes to standard output. `-text` can decode supported compressed files and Hadoop sequence files. Globs are expanded by HDFS, so quote them to prevent the local shell from expanding first.

In [ ]:
%%bash
LAB="/user/$USER/d221"
echo '=== cat one file ==='
hdfs dfs -cat "$LAB/input/raw/employees.csv"
echo '=== head ==='
hdfs dfs -head "$LAB/input/raw/employees.csv"
echo '=== tail ==='
hdfs dfs -tail "$LAB/input/raw/employees.csv"
echo '=== cat a glob ==='
hdfs dfs -cat "$LAB/input/batch/*.txt"

Append local data or standard input to an existing HDFS file. This writes new bytes to a DataNode and updates metadata at the NameNode.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/stdin.txt"
printf '%s\n' 'appended through standard input' | hdfs dfs -appendToFile - "$FILE"
hdfs dfs -cat "$FILE"

## 10. File sizes, directory totals, and counts

`-du` reports logical length and replicated space consumption. With replication 1 they are usually equal. `-count` reports directory count, file count, content size, and optionally quotas.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -du -h "$LAB"
echo
hdfs dfs -du -h -s "$LAB"
echo
hdfs dfs -count -h -q "$LAB"
echo
hdfs dfs -df -h /

## 11. Inspect HDFS blocks

The filesystem shell works with files, not raw block files. `hdfs fsck` asks the NameNode for block metadata and can show each block and its DataNode location. Do not read files by opening DataNode storage directories directly; those are internal implementation details.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/employees.csv"
echo "Configured block size: $(hdfs dfs -stat '%o' "$FILE") bytes"
hdfs fsck "$FILE" -files -blocks -locations

A small file occupies one HDFS block but does not consume the full configured block size. A large file is split into blocks. The client reconstructs it automatically in block order when `-cat` or `-get` is used. There is no normal user command to fetch "block 2" directly.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/employees.csv"
echo 'HDFS checksum:'
hdfs dfs -checksum "$FILE"
echo
echo 'File status including block size and replication:'
hdfs dfs -stat 'length=%b block_size=%o replicas=%r' "$FILE"

### NameNode and DataNode web interfaces

- [NameNode UI](http://localhost:9870) shows cluster capacity, live/dead DataNodes, utilities, and namespace browsing.
- [DataNode UI](http://localhost:9864) shows one DataNode's information and logs.

The NameNode UI is the main cluster-wide view. The DataNode UI is node-specific.

## 12. Download data: `-get` and `-copyToLocal`

These copy HDFS bytes from DataNodes to the local filesystem. `-f` overwrites a local file, `-p` preserves permissions/timestamps where possible, and `-crc` also copies the checksum file.

In [ ]:
%%bash
DOWNLOAD="/tmp/d221-download"
mkdir -p "$DOWNLOAD"
hdfs dfs -get -f "/user/$USER/d221/input/raw/employees.csv" "$DOWNLOAD/employees-copy.csv"
hdfs dfs -copyToLocal -f "/user/$USER/d221/input/raw/stdin.txt" "$DOWNLOAD/stdin-copy.txt"
ls -lh "$DOWNLOAD"
cat "$DOWNLOAD/employees-copy.csv"

`-getmerge` concatenates files from an HDFS directory into one local file. File ordering follows the source listing, so use predictable part-file names.

In [ ]:
%%bash
hdfs dfs -getmerge "/user/$USER/d221/input/batch" /tmp/d221-download/merged.txt
cat /tmp/d221-download/merged.txt

## 13. Copy, move, and rename inside HDFS

`-cp` creates another HDFS file and therefore reads/writes block data. `-mv` within the same HDFS namespace is normally a metadata rename and does not copy block bytes. `-mv` will not overwrite an existing destination.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -cp -f "$LAB/input/raw/employees.csv" "$LAB/input/reference/employees-copy.csv"
hdfs dfs -mv "$LAB/input/reference/employees-copy.csv" "$LAB/archive/employees-archived.csv"
hdfs dfs -ls -R "$LAB/archive"

`-concat TARGET SOURCE...` efficiently joins existing HDFS files into a target when filesystem and block constraints are satisfied. It removes the source files. Use `-getmerge` when the desired result is a local merged file and sources must remain unchanged.

## 14. Empty files and timestamps

`-touchz` creates a zero-byte file and fails if an existing file is non-empty. `-touch` creates a file or updates timestamps. `-setTimes` explicitly changes modification/access time; `-1` leaves a value unchanged.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -touchz "$LAB/_READY"
hdfs dfs -touch "$LAB/_SUCCESS"
hdfs dfs -setTimes "$LAB/_READY" -1 -1
hdfs dfs -ls "$LAB/_READY" "$LAB/_SUCCESS"

## 15. Owners, groups, and permissions

HDFS uses POSIX-like `rwx` permissions for owner, group, and others. For a directory, `r` lists names, `w` creates/deletes children, and `x` traverses the directory. HDFS permissions are enforced by the NameNode. They are independent of local Linux permissions.

The HDFS superuser is usually the Linux account that started the NameNode, not necessarily a user literally named `root`.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -ls -d "$LAB" "$LAB/input/raw"
hdfs dfs -chmod 750 "$LAB/input/raw"
hdfs dfs -chmod u+rw,g+r,o-rwx "$LAB/input/raw/employees.csv"
hdfs dfs -ls -d "$LAB/input/raw"
hdfs dfs -ls "$LAB/input/raw/employees.csv"

Recursive permission changes use `-R`. `-chown` and `-chgrp` usually require the HDFS superuser for ownership changes; the owner may change a file's group to one they belong to. Inspect current values before changing them.

```bash
hdfs dfs -chmod -R 750 /path
hdfs dfs -chown -R USER:GROUP /path
hdfs dfs -chgrp -R GROUP /path
```

### Why `/tmp` is often mode `1777`

Mode `1777` lets everyone create entries but the sticky bit prevents users from deleting entries owned by other users. A shared training cluster may configure HDFS `/tmp` this way. Do not apply permissive modes broadly.

```bash
hdfs dfs -ls -d /tmp
# HDFS superuser only, when provisioning the cluster:
hdfs dfs -mkdir -p /tmp
hdfs dfs -chmod 1777 /tmp
```

### Hive directories

A common Hive setup uses `/user/hive/warehouse` for managed tables and `/tmp/hive` for scratch data. Exact paths come from Hive configuration. Provisioning is an administrator task; broad write permissions are acceptable only for a disposable single-user lab.

```bash
hdfs dfs -ls -d /user/hive/warehouse /tmp/hive
hdfs dfs -stat '%n owner=%u group=%g mode=%a' /user/hive/warehouse /tmp/hive
```

## 16. ACLs and extended attributes

ACLs grant permissions to additional named users/groups beyond the basic owner/group/other model. They work only when ACL support is enabled. Default ACLs on a directory are inherited by new children.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -getfacl "$LAB/input/raw"
echo
echo 'Examples (do not run without a real user/group):'
echo "hdfs dfs -setfacl -m user:analyst:r-x '$LAB/input/raw'"
echo "hdfs dfs -setfacl -m default:group:analytics:r-x '$LAB/input/raw'"
echo "hdfs dfs -setfacl -x user:analyst '$LAB/input/raw'"
echo "hdfs dfs -setfacl -b '$LAB/input/raw'"

Extended attributes attach small key/value metadata to paths. User attributes use the `user.` namespace. Cluster policy may disable or restrict them.

```bash
hdfs dfs -setfattr -n user.description -v 'D221 training data' PATH
hdfs dfs -getfattr -d PATH
hdfs dfs -setfattr -x user.description PATH
```

## 17. Replication

Replication is the requested number of DataNode copies for each block. This lab has only one DataNode, so the practical replication factor is 1. Setting it above the number of available DataNodes leaves blocks under-replicated; it does not create multiple independent copies on the same DataNode.

In [ ]:
%%bash
FILE="/user/$USER/d221/input/raw/employees.csv"
echo "Current replication: $(hdfs dfs -stat '%r' "$FILE")"
hdfs dfs -setrep -w 1 "$FILE"
hdfs fsck "$FILE" -files -blocks -locations

## 18. Delete files and directories

`-rm` removes files, `-rm -r` removes directory trees, and `-rmdir` removes only empty directories. If trash is enabled, normal deletion moves entries into the user's `.Trash`; `-skipTrash` bypasses recovery and should be used cautiously.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfs -cp -f "$LAB/input/raw/stdin.txt" "$LAB/delete-me.txt"
hdfs dfs -mkdir -p "$LAB/empty-dir"
hdfs dfs -rm "$LAB/delete-me.txt"
hdfs dfs -rmdir "$LAB/empty-dir"
hdfs dfs -test -e "$LAB/delete-me.txt" || echo 'delete-me.txt was removed'

Inspect trash rather than assuming it is enabled. `-expunge` permanently removes expired checkpoints according to cluster policy; `-expunge -immediate` is destructive and is not part of this lab.

```bash
hdfs getconf -confKey fs.trash.interval
hdfs dfs -ls -R /user/$USER/.Trash
hdfs dfs -mv /user/$USER/.Trash/Current/PATH RESTORE_PATH
```

## 19. Health and integrity checks

`dfsadmin -report` summarizes DataNodes and capacity. `fsck` checks namespace/block consistency. `-list-corruptfileblocks` lists corrupt paths. These are diagnostics; do not use repair or delete options casually.

In [ ]:
%%bash
LAB="/user/$USER/d221"
hdfs dfsadmin -report
echo
hdfs fsck "$LAB" -files -blocks -locations
echo
hdfs fsck / -list-corruptfileblocks

## 20. Quotas and snapshots: administrator-managed features

Namespace quotas limit file/directory counts; space quotas limit replicated bytes. Snapshots are read-only, point-in-time views of a snapshottable directory. Both require administrative setup.

```bash
hdfs dfs -count -q -h PATH
hdfs dfsadmin -setQuota 10000 PATH
hdfs dfsadmin -setSpaceQuota 10g PATH
hdfs dfsadmin -clrQuota PATH
hdfs dfsadmin -allowSnapshot PATH
hdfs dfs -createSnapshot PATH snapshot_name
hdfs dfs -ls PATH/.snapshot/snapshot_name
hdfs dfs -deleteSnapshot PATH snapshot_name
```

## 21. Troubleshooting by symptom

| Symptom | First checks |
|---|---|
| `Connection refused` | `jps`, `hdfs getconf -confKey fs.defaultFS`, NameNode logs, port 9000 |
| `SafeModeException` | `hdfs dfsadmin -safemode get`, DataNode health; do not force leave until cause is known |
| `Permission denied` | `-ls -d` on every parent, owner/group, `-getfacl`, current user |
| `File exists` | Inspect destination; use a new path or an explicit supported `-f` option |
| Upload stalls/fails | `dfsadmin -report`, DataNode logs, disk space, block locations |
| Under-replicated blocks | Compare requested replication with live DataNode count |
| Local file not found | Remember that `-put` source is local and destination is HDFS |

In [ ]:
%%bash
echo '=== Daemons ==='
jps
echo '=== HDFS endpoint ==='
hdfs getconf -confKey fs.defaultFS
echo '=== NameNode RPC and web ports ==='
ss -lnt | grep -E ':(9000|9870)\b' || true
echo '=== Recent HDFS errors ==='
grep -RniE 'exception|error|failed|corrupt' "$HOME/hadoop-logs" 2>/dev/null | tail -40 || true

## 22. Final challenge

Without copying earlier cells, complete this workflow:

1. Create `/user/$USER/d221/challenge/incoming`.
2. Create three local text files and upload the entire directory.
3. List only `.txt` files recursively.
4. Show size, replication, owner, group, and mode for one file.
5. Inspect its block ID and DataNode location.
6. Change its mode to `640`, then verify it.
7. Copy it inside HDFS, rename the copy, and download the renamed file.
8. Compare local and HDFS checksums or contents.
9. Remove only the challenge directory.

Explain which steps contacted only the NameNode and which transferred bytes through a DataNode.

## 23. Optional lab cleanup

The following removes only the D221 HDFS workspace and local temporary directories. Run it after the instructor confirms that results are no longer needed.

In [ ]:
%%bash
LAB="/user/$USER/d221"
case "$LAB" in
  /user/*/d221) hdfs dfs -rm -r -f "$LAB" ;;
  *) echo "Refusing unexpected HDFS cleanup path: $LAB"; exit 1 ;;
esac
rm -rf /tmp/d221-local /tmp/d221-download
echo 'D221 lab data removed.'

## Command reference

| Task | Command |
|---|---|
| Help | `hdfs dfs -help COMMAND`, `-usage COMMAND` |
| Create | `-mkdir -p`, `-touch`, `-touchz` |
| Upload | `-put`, `-copyFromLocal`, `-moveFromLocal`, `-appendToFile` |
| List/find | `-ls`, `-ls -R`, `-find` |
| Inspect | `-stat`, `-test`, `-du`, `-count`, `-df`, `-checksum` |
| Read | `-cat`, `-head`, `-tail`, `-text` |
| Download | `-get`, `-copyToLocal`, `-getmerge` |
| Organize | `-cp`, `-mv`, `-concat` |
| Permissions | `-chmod`, `-chown`, `-chgrp`, `-getfacl`, `-setfacl` |
| Attributes | `-getfattr`, `-setfattr`, `-setTimes`, `-setrep` |
| Delete | `-rm`, `-rm -r`, `-rmdir`, `-expunge` |
| Blocks/health | `hdfs fsck`, `hdfs dfsadmin -report`, `-safemode get` |